# 🏥 TRIAGE — GRPO Training on Kaggle

**Meta PyTorch OpenEnv Hackathon**

This notebook trains `Qwen2.5-0.5B-Instruct` using **GRPO (Group Relative Policy Optimization)** with 8 independent reward verifiers for hospital crisis decision-making.

**Requirements:** GPU T4 x2 or P100. Enable GPU in Settings → Accelerator.

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth "trl>=0.12" datasets peft accelerate bitsandbytes

In [ ]:
# Cell 2: Import and config
import json, re, random, logging, time, os
from pathlib import Path
from typing import Any

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger('triage_grpo')

# === CONFIG ===
MODEL_NAME = 'Qwen/Qwen3-2B'  # Qwen3.5-2B — best for T4/P100
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 16
LORA_TARGETS = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
NUM_GENERATIONS = 4
MAX_COMPLETION = 128
TEMPERATURE = 0.9
NUM_EPOCHS = 3
BATCH = 1
GRAD_ACCUM = 8
LR = 5e-5
NUM_PROMPTS = 2000
OUTPUT_DIR = './grpo_output'
MERGED_DIR = './grpo_merged'
print('Config loaded.')

In [ ]:
# Cell 3: Self-contained reward verifiers
# All 8 verifiers from triage/rewards/verifiers.py bundled inline

VALID_ACTIONS = frozenset({
    'TRIAGE_PATIENT','ASSIGN_TREATMENT','TRANSFER_TO_ICU','TRANSFER_TO_WARD',
    'ACTIVATE_OVERFLOW','ORDER_MEDICATION','FLAG_POLICY_VIOLATION',
    'OVERRIDE_DECISION','UPDATE_EHR','REQUEST_STAFF','VERIFY_INSURANCE',
})
REQ_KEYS = {'action_type','target_id','priority','reasoning'}
EVIDENCE = [r'P-\d{2,3}',r'patient\s+\d+',r'\d+%',r'\d+/\d+',r'BP\s*\d+',
            r'HR\s*\d+',r'ICU\s+at\s+\d+',r'beds?\s+\d+',r'age\s+\d+',
            r'critical|immediate|urgent|stable']
FORBIDDEN = [(re.compile(p,re.I),n) for p,n in [
    (r'import\s+(os|sys|subprocess)','sys'),(r'exec\s*\(','exec'),
    (r'eval\s*\(','eval'),(r'reward\s*=','hack')]]

def _xjson(t):
    t=t.strip()
    try: return json.loads(t)
    except: pass
    m=re.search(r'```(?:json)?\s*(\{.*?\})\s*```',t,re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except: pass
    m=re.search(r'\{[^{}]*\}',t,re.DOTALL)
    if m:
        try: return json.loads(m.group(0))
        except: pass
    return None

def v_survival(s,c,**k):
    a=s.get('alive_count',0); t=a+s.get('deceased_count',0)
    return a/t if t>0 else 1.0
def v_icu(s,c,**k):
    o=s.get('icu_occupancy',0.5)
    if o<=0.85: return 1.0
    elif o<=0.95: return 1.0-(o-0.85)*5
    else: return max(0,0.5-(o-0.95)*10)
def v_violation(s,c,**k):
    i=s.get('violations_injected',0)
    return min(1,s.get('violations_caught',0)/max(i,1)) if i>0 else 1.0
def v_format(s,c,**k):
    p=_xjson(c)
    if not p or not REQ_KEYS.issubset(p): return 0.0
    if str(p.get('action_type','')).upper() not in VALID_ACTIONS: return 0.0
    try: int(p['target_id'])
    except: return 0.0
    try:
        pr=int(p['priority'])
        if not 1<=pr<=10: return 0.0
    except: return 0.0
    if len(str(p.get('reasoning','')))<10: return 0.0
    return 1.0
def v_reasoning(s,c,**k):
    p=_xjson(c)
    if not p: return 0.0
    r=str(p.get('reasoning',''))
    if len(r)<20: return 0.1
    ev=sum(1 for pt in EVIDENCE if re.search(pt,r,re.I))
    for f in ['i need more','i\'m not sure','let me think']:
        if f in r.lower(): return 0.1
    return min(1.0,0.3+min(0.7,ev*0.15))
def v_speed(s,c,**k):
    l=len(c)
    if l<=400: return 1.0
    elif l<=800: return 1.0-(l-400)*0.001
    else: return max(0.2,0.6-(l-800)*0.0005)
def v_halluc(s,c,**k):
    p=_xjson(c)
    if not p: return 0.5
    ids={int(m.group(1)) for m in re.finditer(r'P-(\d{2,3})',str(p.get('reasoning','')))}
    if not ids: return 1.0
    valid={x.get('id',-1) for x in s.get('patients_summary',[])}
    return 0.0 if ids-valid else 1.0
def v_align(s,c,**k):
    p=_xjson(c)
    if not p: return 0.0
    a=str(p.get('action_type','')).upper()
    o,cr,ct=s.get('icu_occupancy',0.5),s.get('critical_count',0),s.get('crisis_type','')
    tbl={'TRIAGE_PATIENT':1.0 if cr>0 else 0.5,'TRANSFER_TO_ICU':1.0 if o<0.9 and cr>0 else 0.3,
         'ACTIVATE_OVERFLOW':1.0 if o>=0.85 else 0.2,'REQUEST_STAFF':1.0 if ct=='staff_shortage' else 0.4,
         'ORDER_MEDICATION':0.8 if cr>0 else 0.5,'ASSIGN_TREATMENT':0.9 if cr>0 else 0.5}
    return tbl.get(a,0.5)

VERIFIERS=[v_survival,v_icu,v_violation,v_format,v_reasoning,v_speed,v_halluc,v_align]

def compute_rewards(state,completion):
    r={}
    for v in VERIFIERS:
        try: r[v.__name__]=round(float(v(state,completion)),4)
        except: r[v.__name__]=0.0
    r['total']=round(sum(r.values())/len(r),4)
    return r

print(f'Loaded {len(VERIFIERS)} verifiers')

In [ ]:
# Cell 4: Synthetic dataset generator
CRISES = ['mass_casualty','outbreak','equipment_failure','staff_shortage']
AGENTS = ['er_triage','icu_management','pharmacy','cmo_oversight','hr_rostering','it_systems']
ROLES = {'cmo_oversight':'Chief Medical Officer','er_triage':'ER Triage','icu_management':'ICU Management',
         'pharmacy':'Pharmacy','hr_rostering':'HR Rostering','it_systems':'IT Systems'}
ACTS = {'er_triage':'TRIAGE_PATIENT, UPDATE_EHR, ASSIGN_TREATMENT',
        'icu_management':'TRANSFER_TO_ICU, TRANSFER_TO_WARD, ACTIVATE_OVERFLOW',
        'pharmacy':'ORDER_MEDICATION, FLAG_POLICY_VIOLATION','hr_rostering':'REQUEST_STAFF, FLAG_POLICY_VIOLATION',
        'cmo_oversight':'OVERRIDE_DECISION, ACTIVATE_OVERFLOW','it_systems':'UPDATE_EHR, FLAG_POLICY_VIOLATION'}
NAMES=['John Smith','Maria Garcia','Ahmed Hassan','Li Wei','Sarah Johnson','Raj Patel','Emma Wilson']
CONDS=['Cardiac Arrest','Sepsis','Pneumothorax','Hemorrhage','Burns','Fracture','Respiratory Failure','Stroke']

def gen_prompt(rng):
    crisis=rng.choice(CRISES); agent=rng.choice(AGENTS)
    step=rng.randint(1,50); icu_t=rng.randint(20,60)
    icu_pct=rng.randint(40,100) if crisis=='mass_casualty' else rng.randint(30,90)
    icu_o=min(icu_t,int(icu_t*icu_pct/100))
    patients=[]
    for i in range(rng.randint(8,25)):
        st=rng.choices(['CRITICAL','SERIOUS','STABLE'],weights=[.3,.35,.35])[0]
        patients.append({'id':rng.randint(1,99),'name':rng.choice(NAMES),'age':rng.randint(18,90),
                         'status':st,'cond':rng.choice(CONDS),'tri':rng.randint(1,5),'det':round(rng.uniform(0,.5),2)})
    crits=[p for p in patients if p['status']=='CRITICAL'][:5]
    alive=len(patients); dec=rng.randint(0,3); sr=alive/(alive+dec) if alive+dec>0 else 1.0
    vi=rng.randint(0,5); vc=rng.randint(0,vi)
    pl='\n'.join([f"  - P-{p['id']:02d}: {p['name']}, age {p['age']}, {p['status']}, {p['cond']}" for p in crits]) or '  (none)'
    prompt=f"""You are the {agent.upper()} agent in a hospital crisis simulation.

CRISIS: {crisis.upper()}
STEP: {step}/50
ICU OCCUPANCY: {icu_pct}% ({icu_o}/{icu_t} beds)
CRITICAL PATIENTS ({len(crits)} total — top 5):
{pl}
VIOLATIONS INJECTED: {vi} | CAUGHT: {vc}
SURVIVAL RATE: {sr:.1%}

Your role: {ROLES.get(agent,'Staff')}

Decide the single most important action right now. Respond with ONLY valid JSON:
{{
  "action_type": "<one of: {ACTS.get(agent,'TRIAGE_PATIENT')}>",
  "target_id": <patient ID integer or 0>,
  "priority": <integer 1-10>,
  "reasoning": "<1-2 sentences citing specific data>"
}}"""
    state={'alive_count':alive,'deceased_count':dec,'critical_count':len(crits),
           'icu_occupancy':icu_pct/100,'violations_injected':vi,'violations_caught':vc,
           'survival_rate':sr,'crisis_type':crisis,
           'patients_summary':[{'id':p['id'],'status':p['status']} for p in patients[:20]]}
    return {'prompt':prompt,'state':state,'crisis_type':crisis,'agent_type':agent}

rng=random.Random(42)
dataset_raw=[gen_prompt(rng) for _ in range(NUM_PROMPTS)]
print(f'Generated {len(dataset_raw)} prompts')
print(f'Crisis distribution: {dict((c,sum(1 for d in dataset_raw if d["crisis_type"]==c)) for c in CRISES)}')

In [ ]:
# Cell 5: Build reward function for GRPOTrainer
def _state_from_prompt(prompt):
    s={'alive_count':20,'deceased_count':0,'critical_count':0,'icu_occupancy':0.5,
       'violations_injected':0,'violations_caught':0,'survival_rate':1.0,
       'crisis_type':'mass_casualty','patients_summary':[]}
    m=re.search(r'ICU OCCUPANCY:\s*(\d+)%',prompt)
    if m: s['icu_occupancy']=int(m.group(1))/100
    m=re.search(r'CRITICAL PATIENTS\s*\((\d+)',prompt)
    if m: s['critical_count']=int(m.group(1))
    m=re.search(r'VIOLATIONS INJECTED:\s*(\d+)\s*\|\s*CAUGHT:\s*(\d+)',prompt)
    if m: s['violations_injected'],s['violations_caught']=int(m.group(1)),int(m.group(2))
    m=re.search(r'SURVIVAL RATE:\s*(\d+\.?\d*)%',prompt)
    if m: s['survival_rate']=float(m.group(1))/100
    m=re.search(r'CRISIS:\s*(\w+)',prompt)
    if m: s['crisis_type']=m.group(1).lower()
    s['patients_summary']=[{'id':int(x.group(1)),'status':'CRITICAL'} for x in re.finditer(r'P-(\d{2,3})',prompt)]
    s['alive_count']=int(s['survival_rate']*20)
    s['deceased_count']=20-s['alive_count']
    return s

def reward_fn(completions, **kwargs):
    prompts=kwargs.get('prompts',kwargs.get('prompt',['']))
    rewards=[]
    for i,c in enumerate(completions):
        if len(c)>2000 or any(p.search(c) for p,_ in FORBIDDEN):
            rewards.append(0.0); continue
        prompt=prompts[i] if i<len(prompts) else ''
        state=_state_from_prompt(prompt)
        scores=compute_rewards(state,c)
        rewards.append(scores.get('total',0.0))
    return rewards

# Quick test
test_completion=json.dumps({'action_type':'TRIAGE_PATIENT','target_id':42,'priority':1,
                            'reasoning':'P-42 age 75 is critical with 92% ICU occupancy, needs immediate triage.'})
test_scores=compute_rewards({'alive_count':18,'deceased_count':2,'critical_count':5,
    'icu_occupancy':0.92,'violations_injected':2,'violations_caught':1,'survival_rate':0.9,
    'crisis_type':'mass_casualty','patients_summary':[{'id':42,'status':'CRITICAL'}]},test_completion)
print('Test scores:', {k:round(v,2) for k,v in test_scores.items()})

In [ ]:
# Cell 6: Load model + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT, dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, target_modules=LORA_TARGETS,
    lora_alpha=LORA_ALPHA, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
print('Model loaded with LoRA.')

In [ ]:
# Cell 7: Train with GRPO
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig

ds = Dataset.from_dict({'prompt': [r['prompt'] for r in dataset_raw]})

config = GRPOConfig(
    output_dir=OUTPUT_DIR, num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    max_completion_length=MAX_COMPLETION,
    max_prompt_length=MAX_SEQ_LENGTH - MAX_COMPLETION,
    num_generations=NUM_GENERATIONS,
    temperature=TEMPERATURE,
    logging_steps=5, save_steps=100, save_total_limit=3,
    report_to='none', bf16=True, seed=42, log_level='info',
)

trainer = GRPOTrainer(
    model=model, processing_class=tokenizer,
    reward_funcs=reward_fn, args=config, train_dataset=ds,
)

start = time.time()
result = trainer.train()
elapsed = time.time() - start

print(f'\n{"="*60}')
print(f'  Training Complete in {elapsed/60:.1f} min')
print(f'  Steps: {result.global_step}  Loss: {result.training_loss:.4f}')
print(f'{"="*60}')

In [ ]:
# Cell 8: Save and merge
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapters saved to {OUTPUT_DIR}')

try:
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
    print(f'Merged model saved to {MERGED_DIR}')
except Exception as e:
    print(f'Merge note: {e} — LoRA adapters available separately')

In [ ]:
# Cell 9: Quick eval — compare base vs trained
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_prompt = dataset_raw[0]['prompt']
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.3, do_sample=True)
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('=== TRAINED MODEL OUTPUT ===')
print(response)
print()
scores = compute_rewards(_state_from_prompt(test_prompt), response)
print('Scores:', {k: round(v,3) for k,v in scores.items()})

In [ ]:
# Cell 10: Package for download
import shutil
shutil.make_archive('/kaggle/working/triage_grpo_model', 'zip', MERGED_DIR)
print('Model packaged: /kaggle/working/triage_grpo_model.zip')
print('Download from Kaggle Output tab.')